# Ndërtimi i Dataset-it Final të Balancuar

Bazuar në analizën e `explore.ipynb`, ndërtojmë një dataset single-label, të balancuar, me 9 kategori tematike universale (të verifikuara nëpër shumë portale).

## 1. Mapping-u i kategorive

Harta që bashkon variantet/sinonimet e kategorive (nga portale të ndryshme) në 9 kategoritë finale.

In [10]:
import pandas as pd

KATEGORI_MAPPING = {
    # Sport
    'Sport': 'Sport', 'Sporti': 'Sport', 'Premier League': 'Sport',
    'Serie A': 'Sport', 'La Liga': 'Sport', 'Liga e Kampionëve': 'Sport',
    'Superliga': 'Sport', 'Sport vendor': 'Sport',
    
    # Showbiz
    'Showbiz': 'Showbiz', 'Showtime': 'Showbiz', 'Rozë': 'Showbiz',
    'Yjet': 'Showbiz', 'Po Flitet': 'Showbiz', 'Magazina': 'Showbiz',
    
    # Ekonomi
    'Ekonomi': 'Ekonomi', 'Biznes': 'Ekonomi', 'Financa': 'Ekonomi', 'Marketing': 'Ekonomi',
    
    # Kulturë
    'Kulturë': 'Kulturë', 'Muzikë': 'Kulturë',
    
    # Lifestyle
    'Lifestyle': 'Lifestyle',
    
    # Shëndet
    'Shëndet': 'Shëndet', 'Shëndetësi': 'Shëndet', 'Shëndetësi Lajme': 'Shëndet',
    
    # Kuriozitete
    'Kuriozitete': 'Kuriozitete', 'Interesante': 'Kuriozitete', 'Fun': 'Kuriozitete', 'Cool': 'Kuriozitete',
    
    # Politikë
    'Politikë': 'Politikë', 'Politike': 'Politikë',
    
    # Botë
    'Bota': 'Botë', 'Botë': 'Botë', 'Nga Bota': 'Botë',
}

print(f"Total variante të mapuara: {len(KATEGORI_MAPPING)}")
print(f"Kategori finale ({len(set(KATEGORI_MAPPING.values()))}):", sorted(set(KATEGORI_MAPPING.values())))

Total variante të mapuara: 33
Kategori finale (9): ['Botë', 'Ekonomi', 'Kulturë', 'Kuriozitete', 'Lifestyle', 'Politikë', 'Showbiz', 'Shëndet', 'Sport']


## 2. Skanimi i plotë dhe mbledhja e dataset-it të balancuar

(Skanon 5.28M rreshtat në chunks, aplikon mapping-un, mban vetëm artikuj single-label, dhe mbledh deri në një limit të balancuar për kategori.)

In [11]:
LIMIT_PER_KATEGORI = 11000  # bazuar te kategoria më e vogël reale (Ekonomi ~30k, Teknologji hequr)
CHUNK_SIZE = 200000

kategorite_finale = sorted(set(KATEGORI_MAPPING.values()))
mbledhur = {kat: [] for kat in kategorite_finale}

def total_mbledhur():
    return sum(sum(len(df_i) for df_i in v) for v in mbledhur.values())

reader = pd.read_csv('../data/raw/Kosovo-News-Articles.csv', chunksize=CHUNK_SIZE)

for i, chunk in enumerate(reader):
    chunk = chunk.dropna(subset=['category'])
    chunk['category_clean'] = chunk['category'].str.replace(r'Ballina;?', '', regex=True).str.strip(';')
    chunk = chunk[chunk['category_clean'] != '']
    chunk['n_labels'] = chunk['category_clean'].str.split(';').apply(len)
    
    single = chunk[chunk['n_labels'] == 1].copy()
    single['kategoria_finale'] = single['category_clean'].map(KATEGORI_MAPPING)
    single = single.dropna(subset=['kategoria_finale'])
    
    for kat in kategorite_finale:
        aktual = sum(len(df_i) for df_i in mbledhur[kat])
        if aktual < LIMIT_PER_KATEGORI:
            nevojshem = LIMIT_PER_KATEGORI - aktual
            rreshta_kat = single[single['kategoria_finale'] == kat].head(nevojshem)
            if len(rreshta_kat) > 0:
                mbledhur[kat].append(rreshta_kat)
    
    if i % 5 == 0:
        print(f"Chunk {i}, total mbledhur deri tani: {total_mbledhur()}")
    
    if all(sum(len(df_i) for df_i in mbledhur[kat]) >= LIMIT_PER_KATEGORI for kat in kategorite_finale):
        print(f"Mjaftueshëm shembuj u gjetën për të gjitha kategoritë (chunk {i}), ndalojmë.")
        break

df_final = pd.concat([pd.concat(v) for v in mbledhur.values() if v], ignore_index=True)

print("\n" + "="*40)
print("Shape final:", df_final.shape)
print("\nShpërndarja finale:")
print(df_final['kategoria_finale'].value_counts())

Chunk 0, total mbledhur deri tani: 27694
Chunk 5, total mbledhur deri tani: 45423
Chunk 10, total mbledhur deri tani: 67891
Chunk 15, total mbledhur deri tani: 90405
Mjaftueshëm shembuj u gjetën për të gjitha kategoritë (chunk 19), ndalojmë.

Shape final: (99000, 10)

Shpërndarja finale:
kategoria_finale
Botë           11000
Ekonomi        11000
Kulturë        11000
Kuriozitete    11000
Lifestyle      11000
Politikë       11000
Showbiz        11000
Shëndet        11000
Sport          11000
Name: count, dtype: int64


## 3. Ruajtja e dataset-it final

(Ruan `dataset_final.csv` te `data/processed/` — ky është dataset-i që do përdoret për preprocessing, baseline, dhe fine-tuning.)

In [12]:
df_final.to_csv('../data/processed/dataset_final.csv', index=False)
print(f"U ruajt: data/processed/dataset_final.csv ({df_final.shape[0]} rreshta)")

U ruajt: data/processed/dataset_final.csv (99000 rreshta)
